# Settings

## Import Packages

In [1]:
import pandas as pd
import numpy as np
import os
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time
from shapely.wkt import dumps
from shapely.wkt import loads

pd.set_option('display.precision', 4)

In [2]:
def save_geoDataFrame(source_df, filename):
    """
    Save a GeoDataFrame to a CSV file by converting the geometry column to WKT format.

    Parameters:
        source_df (GeoDataFrame): The GeoDataFrame to be saved.
        filename (str): Path to the output CSV file.

    Notes:
        - The geometry column will be converted to WKT strings using shapely.wkt.dumps().
        - The resulting CSV can be restored into a GeoDataFrame using shapely.wkt.loads().
    """
    df = source_df.copy()
    df['geometry'] = df['geometry'].apply(dumps)
    df.to_csv(filename, index=False)

# 2020 United States Census Data
- Instruction:
  1. Download Population Data:
     - Visit the [Census Bureau Data Portal](https://data.census.gov/table?q=P1&g=050XX00US36061,36061$1400000&d=DEC+Demographic+and+Housing+Characteristics)
     - Download the P1 Total Population data for New York County (FIPS: 36061)
	 - Save the downloaded CSV file to the `/raw_data/` directory and rename it as `population_2020.csv`
  2. Download Census Tract Shapefile (Geometry):
	 - Go to the [2020 Census Tract Shapefiles](https://www.census.gov/cgi-bin/geo/shapefiles/index.php?year=2020&layergroup=Census+Tracts)
	 - Select New York as the state and download the shapefile package to the `/raw_data/` directory and unzip the file

In [3]:
population_data = pd.read_csv('raw_data/population_2020.csv')
population_data.columns = population_data.columns.str.replace(r";?\s*New York( County)?;?\s*New York", "", regex=True)
population_df = population_data.T.reset_index()
population_df = population_df.iloc[2:,:]
population_df.columns = ['NAMELSAD', 'population']
population_df

,NAMELSAD,population
2,Census Tract 1,0
3,Census Tract 2.01,"2,012"
4,Census Tract 2.02,"7,266"
5,Census Tract 5,5
6,Census Tract 6,"11,616"
...,...,...
307,Census Tract 309,"8,594"
308,Census Tract 311,12
309,Census Tract 317.03,"5,847"
310,Census Tract 317.04,"10,422"


In [4]:
census_tract = gpd.read_file("raw_data/tl_2020_36_tract/tl_2020_36_tract.shp")

projected = census_tract.to_crs(epsg=2263)
projected['centroid'] = projected.geometry.centroid
centroids = projected.set_geometry('centroid').to_crs(epsg=4326)

census_tract["longitude"] = centroids.geometry.x
census_tract["latitude"] = centroids.geometry.y

manhattan_tracts = census_tract[
    (census_tract['STATEFP'] == '36') & (census_tract['COUNTYFP'] == '061')
].copy()

manhattan_tracts.head(2)

,STATEFP,COUNTYFP,TRACTCE,GEOID,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry,longitude,latitude
1151,36,061,001501,36061001501,15.01,Census Tract 15.01,G5020,S,270812,166085,+40.7075653,-074.0013991,"POLYGON ((-74.0086 40.71139, -74.00835 40.7113...",-74.0016,40.7078
1152,36,061,001600,36061001600,16,Census Tract 16,G5020,S,207381,0,+40.7159568,-073.9932660,"POLYGON ((-73.9975 40.71407, -73.99709 40.7146...",-73.9933,40.7160


In [5]:
combined_population_df = population_df.merge(manhattan_tracts, on='NAMELSAD')
combined_population_df = combined_population_df[['NAMELSAD', 'geometry', 'population']]
save_geoDataFrame(source_df=combined_population_df, filename='census_data_2020.csv')